In [1]:
import spacy
from spacy.language import Language
import os
import json
from pathlib import Path
from pypdf import PdfReader as pr

path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model)

@Language.component('custom_sentencizer')
def create_custom_sentencizer(doc):
    for ent in doc.ents:
        for t in range(len(ent)-1):
            doc[ent.start+t+1].is_sent_start = False
    return doc

sentenciser = scispacy_model.add_pipe('custom_sentencizer', name='custom_sentencizer', after='sentencizer')

c:\Users\jace\miniconda3\envs\scispacy\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [11]:
for ind, file in enumerate(os.listdir("..\\..\\dissertation DLC content\\fermentation_papers_preprocessed")[1:3]):
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"..\\..\\dissertation DLC content\\fermentation_papers_preprocessed\\{filename}", 'r', encoding='utf-8') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf!')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue

    paper_processed = scispacy_model(text)

    all_paper_chemicals = [chem.text for chem in paper_processed.ents if 'CHEMICAL' in chem.label_]

    #no chemicals with which to form a relation
    if len(all_paper_chemicals) == 0:
        print('no chemicals found')
        continue

    all_mic_ents = [(mic.label_, mic.text) for mic in paper_processed.ents if mic.label_ == 'MICROBE_NAME']

    if len(all_mic_ents) == 0:
        print('no microbes found')
        continue

    merge_sentences = []
    last_sent_length = 0
    merge_ents_resolved = []
    merge_update = False

    for sent_raw in paper_processed.sents:
        sent_ents = [(ent.label_, ent.text) for ent in sent_raw.ents]

        if not merge_sentences:
            #if there are entities which are not empty
            if sent_ents:
                merge_sentences.append(sent_raw.text)
                merge_ents_resolved.append(sent_ents)
                merge_update = True

        #but if there is a sentence awaiting merging
        else:
            #if there are any entities to merge with, check that they arent all the same type
            if sent_ents:
                premerged_labels = [label for ent_list in merge_ents_resolved for label, text in ent_list] + [l for l, t in sent_ents]
                if len(set(premerged_labels)) > 1:
                    merge_ents_resolved.append(sent_ents)
                    merge_sentences.append(sent_raw.text)
                    merge_update = True

                while len(merge_sentences) > 3:
                    print("reducing sentences")
                    merge_sentences.pop(0)
                    merge_ents_resolved.pop(0)
                    merge_update = True

        all_merge_ents = [ent for ent_list in merge_ents_resolved for ent in ent_list]
      
        #checking to see if the merged ents can be written to the file, or if we should just keep merging until theres enough to form a 'relation'
        if (len(merge_sentences) > 2 or len(all_merge_ents) >= 3) and merge_update:
            microbes = list(set([mic for label,mic in all_merge_ents if 'MICROBE' in label]))
            no_microbes = len(microbes)
            chemicals = list(set([text for label, text in all_merge_ents if 'CHEMICAL' in label])) #the types of chemicals identified
            no_chemicals = len(chemicals)

            condition1 = no_microbes >= 1 and no_chemicals >= 1 #MICROBE CONSUMES/PRODUCES CHEMICAL
            if condition1:
                print("processing merge")
                print(microbes, chemicals)

                merge_update = False

        print("length:", len(merge_sentences))

length: 0
length: 0
length: 1
length: 1
length: 2
length: 3
reducing sentences
processing merge
['K. fragilis', 'Lactobacillus sanfrancisco', 'Aspergillus oryzae', 'Monascus purpureus', 'Candida milleri', 'Xanthomonas campestris', 'Yarrowia lipolytica', 'Eremothecium ashbyi', 'Lactobacillus bulgaricus', 'Saccharomyces carlsbergensis', 'Penicillium roqueforti', 'Gluconobacter suboxydans', 'Phaffia rhodozyma', 'A. niger', 'C. glutamicum', 'S. cerevisiae', 'Propionibacterium shermanii', 'Leuconostoc mesenteroides', 'Kluyveromyces fragilis', 'Endothia parasitica', 'Methylophilus methylotrophus', 'NY. Application', 'Clostridium acetobutylicum', 'Cephalosporium acremonium', 'X. campestris', 'Pseudomonas denitrificans', 'Penicillium chrysogenum', 'Trichoderma reesei', 'Penicillium camemberti', 'Streptococcus thermophilus', 'Candida utilis', 'Saccharomyces cerevisiae', 'Saccharomyces rouxii', 'S. lipolytica'] ['Citric acid', 'sour', 'Vitamins', 'methanol', 'neomycin', 'Riboflavin', 'butanol', 